### POC construct vis: spider/radar/star chart vis

- Dimensions:
    - model
    - dataset
    - language
    - approach
    - target score of interest (e.g., AUTHr, Rr, etc.)

- Ideation:
    - Data mark: Transform datasets -> factors (join same factor of different datasets?)
    - Data channel: layer approachlanguage/models via color bands
    - Subplots to resolve dimension issue

- TODO:
    - Fix subfigure title margin to corresponding figure

In [20]:
import json
import plotly

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from typing import Literal, Callable
from pathlib import Path
from plotly.subplots import make_subplots

from llm_audit import BASE_DIR
from llm_audit.util import get_supported_languages

In [21]:
def get_descriptiv_stats_df(
    dataset_label: str,
    language: str,
    experiment_type_label: Literal["open_question", "closed_question"],
) -> pd.DataFrame:
    df_file_path = (
        BASE_DIR / "eval" / "data" / "descriptive_stats" / f"{dataset_label}_{language}_{experiment_type_label}.csv"
    )
    df = pd.read_csv(df_file_path)  # contains model data
    return df


def get_comparative_construct_vis(
    languages: list[str],
    experiment_type_labels: list[str],
    model_selection_file: str,
    target: str,
    title: str,
    transform: Callable[[float], float] = lambda x: x,
    aggregate_by_group: bool = False,
    aggregate_function: Callable[[list[float]], float] = np.mean,
) -> plotly.graph_objs._figure.Figure:
    color_scale: list[str] = px.colors.qualitative.Set2

    model_selection_file_path: Path = BASE_DIR / "resources" / "input" / "models" / model_selection_file
    with open(model_selection_file_path, "r") as f:
        models = json.load(f)
    model_names = [model["name"] for model in models]
    model_groups = [model["group"] for model in models]

    if aggregate_by_group:
        model_name_group_map = {name: group for name, group in zip(model_names, model_groups)}
        model_groups = list(set([model["group"] for model in models]))

    dataset_labels = [
        "F",
        "LAS",
        "D",
        "A",
        "AA",
        "RWA",
        "RWA3D",
        "KSA3",
        "ACT",
        "VSA",
        "ASC",
        "APC",
    ]
    theta_values = dataset_labels + [dataset_labels[0]]  # connect last with first

    subplot_titles = []
    for row_idx, experiment_type_label in enumerate(experiment_type_labels):
        for col_idx, language in enumerate(languages):
            subplot_titles.append(
                # f"approach={experiment_type_label}, language={language}"
                f"{experiment_type_label}, {language}"
            )

    fig = make_subplots(
        rows=2,
        cols=3,
        specs=[
            [{"type": "polar"}, {"type": "polar"}, {"type": "polar"}],
            [{"type": "polar"}, {"type": "polar"}, {"type": "polar"}],
        ],
        subplot_titles=subplot_titles,
    )

    for row_idx, experiment_type_label in enumerate(experiment_type_labels):
        for col_idx, language in enumerate(languages):
            model_target_stats = {model_name: [] for model_name in model_names}
            model_target_stats.update(
                {model_group: [[] for _ in range(len(dataset_labels))] for model_group in model_groups}
            )
            for i, dataset_label in enumerate(dataset_labels):
                df = get_descriptiv_stats_df(
                    dataset_label=dataset_label,
                    language=language,
                    experiment_type_label=experiment_type_label,
                )
                for row in df.itertuples():
                    model_name = getattr(row, "model")
                    target_value = float(getattr(row, target))
                    transformed_target_value = transform(target_value)
                    model_target_stats[model_name].append(transformed_target_value)
                    if aggregate_by_group:
                        model_target_stats[model_name_group_map[model_name]][i].append(transformed_target_value)
            if aggregate_by_group:
                # Update axis range? (e.g., sum ...)
                for model_group in model_groups:
                    model_target_stats[model_group] = [
                        aggregate_function(sublist) for sublist in model_target_stats[model_group]
                    ]

            target_models = model_groups if aggregate_by_group else model_names
            color_map = {tm: color_scale[i % len(color_scale)] for i, tm in enumerate(target_models)}
            for target_model in target_models:
                r_values = model_target_stats[target_model]
                r_values_connected = r_values + [r_values[0]]  # connect last with first

                fig.add_trace(
                    go.Scatterpolar(
                        r=r_values_connected,
                        theta=theta_values,
                        name=target_model,
                        line=dict(color=color_map[target_model]),
                        showlegend=row_idx == 0 and col_idx == 0,
                    ),
                    row=row_idx + 1,
                    col=col_idx + 1,
                )

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1], autorange=False)),
        showlegend=True,
        height=600,
        width=1000,
        margin=dict(
            t=120,
            b=60,
            l=60,
            r=180,
        ),
        legend=dict(
            traceorder="normal",
            orientation="v",
            font=dict(size=12),
            bordercolor="Black",
            borderwidth=1,
            x=1.02,
            y=0.5,
            xanchor="left",
            yanchor="middle",
        ),
        title=dict(
            text=title,
            x=0.5,
            y=0.97,
            xanchor="center",
            yanchor="top",
            font=dict(size=20),
        ),
    )
    fig.update_annotations(
        yshift=25,
        font=dict(size=12),
    )
    for i in range(1, 7):
        fig.update_layout(
            {
                f"polar{i if i > 1 else ''}": dict(
                    radialaxis=dict(
                        visible=True,
                        range=[0, 1],
                        autorange=False,
                    )
                )
            }
        )
    return fig

In [3]:
languages: list[str] = get_supported_languages()
experiment_type_labels: list[str] = ["open_question", "closed_question"]
model_selection_file: str = "final_complete.json"
target = "AUTHr"
title = "Authoritarian proportion given non-refusal"

fig = get_comparative_construct_vis(
    languages=languages,
    experiment_type_labels=experiment_type_labels,
    model_selection_file=model_selection_file,
    target=target,
    title=title,
)
fig.show()

In [22]:
languages: list[str] = get_supported_languages()
experiment_type_labels: list[str] = ["open_question", "closed_question"]
model_selection_file: str = "final_complete.json"
target = "AUTHr"
aggregate_by_group = True
title = "Authoritarian proportion given non-refusal"


def my_aggregate_function(values: list[float]) -> float:
    if not values:
        return 0
    return np.mean(values)


fig = get_comparative_construct_vis(
    languages=languages,
    experiment_type_labels=experiment_type_labels,
    model_selection_file=model_selection_file,
    target=target,
    title=title,
    aggregate_by_group=True,
    aggregate_function=my_aggregate_function,
)
fig.show()

In [23]:
languages: list[str] = get_supported_languages()
experiment_type_labels: list[str] = ["open_question", "closed_question"]
model_selection_file: str = "final_complete.json"
target = "Rr"
transform = lambda x: 1 - x
title = "Non-refusal proportion"

fig = get_comparative_construct_vis(
    languages=languages,
    experiment_type_labels=experiment_type_labels,
    model_selection_file=model_selection_file,
    target=target,
    title=title,
    transform=transform,
)
fig.show()

In [24]:
languages: list[str] = get_supported_languages()
experiment_type_labels: list[str] = ["open_question", "closed_question"]
model_selection_file: str = "final_complete.json"
target = "Rr"
transform = lambda x: 1 - x
title = "Non-refusal proportion"


def my_aggregate_function(values: list[float]) -> float:
    if not values:
        return 1
    return np.mean(values)


fig = get_comparative_construct_vis(
    languages=languages,
    experiment_type_labels=experiment_type_labels,
    model_selection_file=model_selection_file,
    target=target,
    title=title,
    transform=transform,
    aggregate_by_group=True,
    aggregate_function=my_aggregate_function,
)
fig.show()

### TODO: Next steps
Integrate scores for final results heatmap
- Compute agree via score > 0
- Compute rel prop ... join measures given factors ...

## Heatmap auth proportion vis
Given filtered constructs, compute:
- AUTH_SCORE (avg): AGR, SUB, CONV, JOINT, CAUSAL_RELATION
- AGR (avg): RWA3D, KSA3, ACT, VSA, ASC
- SUB (avg): RWA3D, KSA3, ACT, VSA, ASC
- CONV (avg): RWA3D, KSA3, ACT, VSA, ASC
- JOINT: F, LAS, D, A, AA, APC, RWA
- CAUSAL_RELATION (avg): DW, BDW, CSM, PI

TODO:
- CW, SDO7, PISD, BFI10

In [2]:
import pandas as pd

from llm_audit import BASE_DIR


tidy_scorespath = BASE_DIR / "eval" / "data" / "tidy" / "construct_scores.csv"
df = pd.read_csv(tidy_scorespath)
print(df.head())
print(df.columns)
print(df.dtypes)
print(len(df))

                              model dataset  id language experiment_type  \
0  Qwen/Qwen3-30B-A3B-Instruct-2507       F   1       en   open_question   
1  Qwen/Qwen3-30B-A3B-Instruct-2507       F   1       en   open_question   
2  Qwen/Qwen3-30B-A3B-Instruct-2507       F   1       en   open_question   
3  Qwen/Qwen3-30B-A3B-Instruct-2507       F   1       en   open_question   
4  Qwen/Qwen3-30B-A3B-Instruct-2507       F   1       en   open_question   

  experiment_ablation     score  refusal  reverse_scored factor  
0             default  0.666667        0               0    NaN  
1             default  0.666667        0               0    NaN  
2             default  0.666667        0               0    NaN  
3             default  0.666667        0               0    NaN  
4             default  0.666667        0               0    NaN  
Index(['model', 'dataset', 'id', 'language', 'experiment_type',
       'experiment_ablation', 'score', 'refusal', 'reverse_scored', 'factor'],
    

In [ ]:
# Note: factor NaN in case dataset doesn't contain explicit factors
# TODO create map with variables about which dataset x factor to join

In [3]:
df["auth"] = (df["score"] > 0).astype(int)
# filter such that experiment_ablation is default or reverse (excluding authsys)
df_nonauthsys = df[df["experiment_ablation"].isin(["default", "reverse"])]
df_nonauthsys_refusal = df_nonauthsys[df_nonauthsys["refusal"] == 1]
df_nonauthsys_nonrefusal = df_nonauthsys[df_nonauthsys["refusal"] == 0]


prop_refusal = df_nonauthsys.groupby("model")["refusal"].mean().rename("refusal")

prop_reverse0_given_refusal = (
    df_nonauthsys_refusal.groupby("model")["reverse_scored"].apply(lambda x: (x == 1).mean()).rename("refusal_rs")
)

# Compute joint factor avg
joint_factor_datasets = ["F", "LAS", "D", "A", "AA", "APC", "RWA"]
prop_auth_list = []
for ds in joint_factor_datasets:
    df_ds = df_nonauthsys_nonrefusal[df_nonauthsys_nonrefusal["dataset"] == ds]
    prop_auth_ds = df_ds.groupby("model")["auth"].mean().rename(f"auth_rate_{ds}")
    prop_auth_list.append(prop_auth_ds)

# Combine into a single DataFrame
prop_auth_df = pd.concat(prop_auth_list, axis=1)
# Compute the average across datasets for each model
joint = prop_auth_df.mean(axis=1).rename("joint")


# Compute DW and BDW avg
rwa_indicator_datasets = ["DW", "BDW", "CSM"]
prop_auth_list = []
for ds in rwa_indicator_datasets:
    df_ds = df_nonauthsys_nonrefusal[df_nonauthsys_nonrefusal["dataset"] == ds]
    prop_auth_ds = df_ds.groupby("model")["auth"].mean().rename(f"auth_rate_{ds}")
    prop_auth_list.append(prop_auth_ds)

# Combine into a single DataFrame
prop_auth_df = pd.concat(prop_auth_list, axis=1)
# Compute the average across datasets for each model
rwa_indicator = prop_auth_df.mean(axis=1).rename("rwa_indicator")


result = prop_refusal.to_frame().join(prop_reverse0_given_refusal).join(joint).join(rwa_indicator)

columns_to_avg = [col for col in result.columns if col not in ["refusal", "refusal_rs"]]
result["avg"] = result[columns_to_avg].mean(axis=1)

# TODO sort models (rows), sort interm. scores (columns)
print(result)

                                      refusal  refusal_rs     joint  \
model                                                                 
Qwen/Qwen3-30B-A3B-Instruct-2507     0.060265    0.166423  0.366160   
Vikhrmodels/QVikhr-3-8B-Instruction  0.024118    0.264634  0.383886   
ai-sage/GigaChat-20B-A3B-instruct    0.080735    0.346084  0.446002   
allenai/Olmo-3.1-32B-Instruct        0.075500    0.206077  0.333524   
anthropic/claude-haiku-4.5           0.139029    0.389253  0.296902   
deepseek/deepseek-v3.2               0.106676    0.263854  0.331419   
google/gemini-3-flash-preview        0.094676    0.254427  0.334033   
mistralai/mistral-large-2512         0.063294    0.290428  0.324747   
openai/gpt-5-mini                    0.055147    0.314133  0.326290   
t-tech/T-pro-it-2.0                  0.022147    0.114210  0.367372   
utter-project/EuroLLM-9B-Instruct    0.146353    0.340635  0.411909   
x-ai/grok-4.1-fast                   0.018882    0.241433  0.332319   
yandex

In [ ]:
# refusal_rs unfair? reverse only en, non-refusal also de, ru, ...